# GPUBMA — exact exhaustive 2^30 BMA on `panel_30_center15` (Google Colab, NVIDIA A100)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Favioleiva/gpubma/blob/main/notebooks/GPUBMA_A100_p30.ipynb)

Runs the **validated exact GPUBMA enumerator** (streamed direct batching,
float64 only, deterministic reductions, SHA-256-gated checkpoint/resume —
`docs/ADR_0001_GPU_ENUMERATOR.md`, `docs/FWL_BLOCK_FORMULATION.md`) on the
frozen **`panel_30_center15`** benchmark: p = 30 candidates
(**15 correlated true regressors x1–x15 + 15 structural-zero proxies
x16–x30**), n = 2,000, shrink (Stata-verified) convention, g = 2000,
beta-binomial(1, 1) model prior, **all 2^30 = 1,073,741,824 models exactly**
— no MCMC, no sampling, no pruning, no truncation, no early stopping.

**How to run**
1. Open this notebook in Colab (badge above, or `File → Open notebook →
   GitHub`).
2. Runtime → Change runtime type → **A100 GPU** (the A100 40 GB runtime is
   the validated target; do **not** run this on CPU or a small GPU).
3. Run the cells top to bottom: bootstrap → input validation → **mandatory
   smoke test** → (optionally mount Drive) → set `RUN_FULL_EXACT_P30 = True`
   in the config cell → run the expensive cell → validate → download the
   compact results ZIP.

**Cost warning.** The full run enumerates 1,073,741,824 models and consumes
Colab compute units. **No runtime claim exists yet for this benchmark** —
the previous ~95 s A100 figure is *historical* and belongs to the OLD,
easier `panel_30` sparse benchmark (n = 1,000); do not attribute it to
`panel_30_center15` (n = 2,000).

**Storage modes — Google Drive is OPTIONAL and disabled by default.**
Everything the run needs (code, notebook, dataset, metadata) comes from the
public GitHub repository; no paid Drive plan, no Drive mount, no upload,
no GitHub credential, and no token are required.

- *Default, free mode* (`USE_GOOGLE_DRIVE = False`):
  GitHub inputs → Colab local runtime (`/content`) → manual checkpoint /
  result downloads. Colab local storage is **ephemeral**: checkpoints
  survive cell reruns within the same active runtime, but are lost if the
  runtime is destroyed — protect progress by downloading a checkpoint
  bundle (cell below) and re-uploading it after a reset.
- *Optional persistence mode* (`USE_GOOGLE_DRIVE = True`):
  checkpoints/results live on your Drive and a destroyed session resumes
  automatically after Run all.

**Scientific expectation.** The benchmark is *designed* so that proxies
compete with true variables (proxy–primary correlations 0.79–0.89).
Posterior mass may legitimately spread across observationally similar
models, and **the exact true model need not have the highest PMP** —
family-level recovery (true+proxy groups) is the scientifically meaningful
outcome. See `reports/panel_30_center15_dgp_validation.md`.


In [ ]:
# [CONFIG] User-editable configuration — defaults target the validated
# A100 40 GB workflow. Tags per setting: [correctness] [precision]
# [gpu-mem] [host-mem] [runtime] [ckpt-compat] [output-size].
#
# There is NO setting that switches the method to MCMC/sampling/pruning/
# truncation/approximation — the enumeration is always exact.

GITHUB_REPO_URL = 'https://github.com/Favioleiva/gpubma.git'
# Exact public commit this run is pinned to. [correctness, ckpt-compat]
PINNED_COMMIT = 'ef9d68af462564b5cbdce268be1ff2b43317dd24'
REPO_DIR = '/content/gpubma'

# ---- method-critical (do not change) --------------------------------------
DTYPE = 'float64'    # fixed project policy (CLAUDE.md rule 6): the GPU
                     # enumerator is float64-only. [correctness, precision]
P = 30               # 2^30 = 1,073,741,824 models, enumerated exactly.
DATASET_REL = 'data/synthetic/panel_30_center15.parquet'
METADATA_REL = 'data/synthetic/panel_30_center15_metadata.json'
EXPECTED_SEED = 20260724
MODEL_PRIOR = ('betabinomial', 1.0, 1.0)   # [correctness]
# g = max(n, p^2) = 2000 for n = 2000 — resolved after loading. [correctness]

# ---- performance (safe to tune) --------------------------------------------
MAX_CHUNK = 1 << 16          # models per GPU batch. [gpu-mem, runtime]
                             # Resume is rank-based, so changing it between
                             # sessions is checkpoint-COMPATIBLE.
VRAM_BUDGET_FRACTION = 0.25  # share of free VRAM used for batch buffers. [gpu-mem]

# ---- storage / persistence --------------------------------------------------
# Everything runs from the public GitHub clone + Colab local storage.
# No Drive plan, mount, upload, credential, or token is needed.
LOCAL_RUN_ROOT = '/content/GPUBMA_center15_run'  # outside the git clone
OUTPUT_DIRNAME = 'results_p30_center15'
TMP_DIRNAME = 'tmp'
CHECKPOINT_DIRNAME = 'checkpoints'
CHECKPOINT_EVERY_S = 60.0    # checkpoint cadence [runtime (negligible)]
AUTO_RESUME = True           # resume from a matching checkpoint when found

# ---- OPTIONAL Google Drive convenience (disabled by default) -----------------
# Only provides persistence across DESTROYED Colab sessions. Never required
# for loading the dataset or completing the run. When False, the notebook
# never mounts Drive, never prompts for authorization, and never touches
# /content/drive.
USE_GOOGLE_DRIVE = False
OPTIONAL_DRIVE_BASE = 'GPUBMA_center15_run'      # under MyDrive if enabled

# ---- passes and outputs -------------------------------------------------------
RUN_PASS2 = True                # exact second sweep: top-K table, family joint
                                # posterior, exact true-model rank, sign
                                # probabilities, densities. [runtime]
COMPUTE_COEF_DENSITIES = True   # exact conditional density grids (PASS2)
                                # [runtime, output-size]
GRID_POINTS = 257               # density grid resolution [output-size]
TOP_K_MODELS = 10000            # retained top models [output-size; tiny memory]
PROGRESS_EVERY_S = 30.0         # progress cadence [runtime (negligible)]

# ---- smoke test and gates ----------------------------------------------------
SMOKE_P = 15                 # reduced size validated in the local phase
SMOKE_N_RANDOM_MODELS = 64   # deterministic 'random' model IDs
SMOKE_RNG_SEED = 20260724    # deterministic seed for those IDs
EXPERT_OVERRIDE_SKIP_SMOKE_GATE = False   # DANGER: bypasses the CPU/GPU
                                          # parity gate. Leave False.
RUN_FULL_EXACT_P30 = False   # <<< the expensive cell refuses to start until
                             #     YOU set this to True, on purpose.
print('config loaded; RUN_FULL_EXACT_P30 =', RUN_FULL_EXACT_P30)


In [ ]:
# [GPU] Hardware check — fails clearly without CUDA; warns if not an A100.
import subprocess, sys

try:
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(smi.stdout or smi.stderr)
except FileNotFoundError:
    print('nvidia-smi not found — no NVIDIA driver in this runtime')

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Select an A100 GPU runtime: Runtime -> '
        'Change runtime type -> A100 GPU. DO NOT run this workflow on CPU.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {GPU_NAME} ({GPU_VRAM_GIB:.1f} GiB VRAM)')
q = subprocess.run(['nvidia-smi', '--query-gpu=driver_version',
                    '--format=csv,noheader'], capture_output=True, text=True)
DRIVER_VERSION = q.stdout.strip() or 'unknown'
print('driver:', DRIVER_VERSION, '| torch CUDA:', torch.version.cuda)

# real float64 sanity op (GPUBMA is float64-only)
a = torch.randn(512, 512, dtype=torch.float64, device='cuda')
assert (a @ a.T).dtype == torch.float64
if 'A100' not in GPU_NAME:
    print('WARNING: this is NOT an A100. The A100 40 GB runtime is the '
          'validated target; smaller GPUs will be slow and may exhaust VRAM. '
          'Proceed only if you know what you are doing.')
else:
    print('A100 detected — validated target runtime.')


In [ ]:
# [BOOTSTRAP] Clone the public repository at the pinned commit (idempotent),
# install gpubma, print the software environment. A stale or dirty clone is
# detected and reported — never silently overwritten.
import importlib, importlib.metadata, os, pathlib, subprocess, sys

def sh(*cmd, cwd=None, check=True):
    r = subprocess.run(list(cmd), cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed: {cmd}\n{r.stdout}\n{r.stderr}')
    return r.stdout.strip()

repo = pathlib.Path(REPO_DIR)
if repo.exists():
    if not (repo / '.git').exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a git clone — '
                           'remove it manually and rerun this cell.')
    dirty = sh('git', '-C', REPO_DIR, 'status', '--porcelain')
    if dirty:
        raise RuntimeError(
            'the existing clone has local uncommitted changes; refusing to '
            'overwrite them. Inspect or delete /content/gpubma manually:\n'
            + dirty)
    sh('git', '-C', REPO_DIR, 'fetch', 'origin')
    print('existing clean clone found — reusing it')
else:
    print(sh('git', 'clone', GITHUB_REPO_URL, REPO_DIR))

sh('git', '-C', REPO_DIR, 'checkout', '--detach', PINNED_COMMIT)
RESOLVED_COMMIT = sh('git', '-C', REPO_DIR, 'rev-parse', 'HEAD')
assert RESOLVED_COMMIT.startswith(PINNED_COMMIT), (RESOLVED_COMMIT, PINNED_COMMIT)
print('pinned public commit:', RESOLVED_COMMIT)

required = ['notebooks/GPUBMA_A100_p30.ipynb', DATASET_REL, METADATA_REL,
            'src/gpubma/gpu/enumerator.py', 'src/gpubma/cpu/enumeration.py',
            'src/gpubma/priors/model_priors.py']
for rel in required:
    assert (repo / rel).exists(), f'pinned commit is missing {rel}'
print('pinned tree contains the notebook, dataset, metadata, and modules')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                REPO_DIR, 'psutil'], check=True)
if str(repo / 'src') not in sys.path:
    sys.path.insert(0, str(repo / 'src'))
importlib.invalidate_caches()

def _ver(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'not installed'

import torch
print('Python :', sys.version.split()[0])
print('CUDA   :', torch.version.cuda, '| driver:', DRIVER_VERSION)
for pkg in ('gpubma', 'numpy', 'pandas', 'pyarrow', 'scipy', 'torch'):
    print(f'{pkg:8s}:', _ver(pkg))
print('cupy    :', _ver('cupy-cuda12x'), '(not used by gpubma — torch only)')


In [ ]:
# [STORAGE] Resolve run directories. Default: Colab local runtime storage
# under /content (no Drive). Drive is mounted ONLY if you explicitly set
# USE_GOOGLE_DRIVE = True in the config cell.
import pathlib

if USE_GOOGLE_DRIVE:
    from google.colab import drive   # imported only in this branch
    drive.mount('/content/drive')
    WORK_DIR = pathlib.Path('/content/drive/MyDrive') / OPTIONAL_DRIVE_BASE
    print('optional Drive mode: checkpoints/results persist across '
          'destroyed sessions on your Drive')
else:
    WORK_DIR = pathlib.Path(LOCAL_RUN_ROOT)
    print('default free mode: everything stays in Colab local storage.\n'
          'PERSISTENCE NOTE: local checkpoints survive cell reruns within '
          'this active runtime,\nbut are LOST if Colab destroys or resets '
          'the runtime. Google Drive is optional and\nonly adds persistence '
          'across destroyed sessions — it is NOT needed to load the\n'
          'dataset or complete the run. To protect progress without Drive, '
          'periodically run\nexport_checkpoint_bundle() + '
          'download_checkpoint_bundle() (cell below).')

# runtime files live OUTSIDE the git clone so they can never be staged
assert not str(WORK_DIR).startswith(str(pathlib.Path(REPO_DIR))), \
    'work dir must be outside the cloned repository'
OUT_DIR = WORK_DIR / OUTPUT_DIRNAME
RESULTS_DIR = OUT_DIR / 'results'
REPORTS_DIR = OUT_DIR / 'reports'
TMP_DIR = WORK_DIR / TMP_DIRNAME
CKPT_DIR = WORK_DIR / CHECKPOINT_DIRNAME
for d in (OUT_DIR, RESULTS_DIR, REPORTS_DIR, TMP_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)
PASS1_CKPT = CKPT_DIR / 'enum_p30_center15.ckpt.npz'
PASS2_CKPT = CKPT_DIR / 'pass2_p30_center15.ckpt.npz'
PASS1_JSON = RESULTS_DIR / 'panel_30_center15_pass1_raw.json'
print('work dir      :', WORK_DIR)
print('results dir   :', RESULTS_DIR)
print('checkpoint dir:', CKPT_DIR)
print('temp dir      :', TMP_DIR)


In [ ]:
# [VALIDATE INPUT] Load the canonical dataset + metadata from the pinned
# public clone and verify every documented property BEFORE any enumeration.
import hashlib, json, math, pathlib, time
import numpy as np
import pandas as pd

t_load0 = time.perf_counter()
DATA_PARQUET = pathlib.Path(REPO_DIR) / DATASET_REL
META_PATH = pathlib.Path(REPO_DIR) / METADATA_REL
META = json.loads(META_PATH.read_text())

PARQUET_SHA256 = hashlib.sha256(DATA_PARQUET.read_bytes()).hexdigest()
METADATA_SHA256 = hashlib.sha256(META_PATH.read_bytes()).hexdigest()
assert PARQUET_SHA256 == META['parquet_sha256'], (
    'Parquet SHA-256 mismatch — the clone does not contain the frozen '
    'canonical artifact')

DF = pd.read_parquet(DATA_PARQUET)
n = len(DF)
TRUE_VARS = [f'x{j}' for j in range(1, 16)]
PROXY_VARS = [f'x{j}' for j in range(16, 31)]
EXPECTED_COLS = ['individual_id', 'period', 'y'] + \
    [f'x{j}' for j in range(1, 31)] + ['w1', 'w2']

checks = []
def chk(name, ok, detail=''):
    checks.append((name, bool(ok), detail))
    if not ok:
        raise AssertionError(f'input validation FAILED: {name} ({detail})')

chk('generator seed = 20260724', META['seed'] == EXPECTED_SEED, META['seed'])
chk('n = 2,000', n == 2000, n)
chk('35 columns, expected names/order', list(DF.columns) == EXPECTED_COLS)
chk('30 candidate regressors', META['n_candidate_regressors'] == 30)
chk('15 true regressors', META['true_variables'] == TRUE_VARS)
chk('15 proxy regressors', META['proxy_variables'] == PROXY_VARS)
chk('dtypes (int32 ids, float64 data)',
    str(DF['individual_id'].dtype) == 'int32'
    and str(DF['period'].dtype) == 'int32'
    and all(str(DF[c].dtype) == 'float64' for c in EXPECTED_COLS[2:]))
chk('no missing values', not DF.isna().any().any())
chk('no non-finite values', np.isfinite(DF.to_numpy(np.float64)).all())

y = DF['y'].to_numpy(np.float64)
X30 = DF[[f'x{j}' for j in range(1, 31)]].to_numpy(np.float64)
A = np.column_stack([np.ones(n), DF[['w1', 'w2']].to_numpy(np.float64)])
Q, _ = np.linalg.qr(A)
D_true = np.column_stack([np.ones(n), DF[['w1', 'w2']].to_numpy(np.float64),
                          DF[TRUE_VARS].to_numpy(np.float64)])
Qt, _ = np.linalg.qr(D_true)
resid = y - Qt @ (Qt.T @ y)
yc = y - y.mean()
R2 = float(1.0 - (resid @ resid) / (yc @ yc))
SNR = R2 / (1.0 - R2)
chk('realized R^2 in [0.695, 0.705]', 0.695 <= R2 <= 0.705, f'{R2:.6f}')
chk('R^2 equals metadata value',
    abs(R2 - META['noise_calibration']['realized_r2']) < 1e-9)
chk('SNR in validated band [2.28, 2.39]', 2.28 <= SNR <= 2.39, f'{SNR:.4f}')
X_r = X30 - Q @ (Q.T @ X30)
chk('rank(X) = 30', np.linalg.matrix_rank(X30) == 30)
chk('rank(residualized X) = 30', np.linalg.matrix_rank(X_r) == 30)
eig = np.linalg.eigvalsh(X_r.T @ X_r)
COND = float(eig[-1] / eig[0])
chk('residualized Gram condition in validated range [50, 85]',
    50.0 <= COND <= 85.0, f'{COND:.2f}')

# validated shrink-convention inputs for the enumerator
Y_R = y - Q @ (Q.T @ y)
CONV = dict(df_resid=n - 1, tss_norm=float(yc @ yc), k_always=2)
G = float(max(n, P * P))          # = 2000 (benchmark rule max(n, p^2))
from gpubma.priors.model_priors import log_model_prior_function
LOG_PRIOR, PRIOR_DESC = log_model_prior_function(MODEL_PRIOR, P)
N_MODELS = 1 << P
TRUE_MASK = (1 << 15) - 1          # x1..x15 = bits 0..14
T_DATA_LOAD_S = time.perf_counter() - t_load0

print(f'{"check":55s} status')
for name, ok, detail in checks:
    print(f'{name:55s} {"PASS" if ok else "FAIL"} {detail}')
print(f'\nparquet  sha256: {PARQUET_SHA256}')
print(f'metadata sha256: {METADATA_SHA256}')
print(f'g = {G:.0f}, prior = {PRIOR_DESC}, models = {N_MODELS:,}, '
      f'load+validate {T_DATA_LOAD_S:.1f} s')
INPUT_VALIDATED = True


In [ ]:
# [CHECKPOINT TOOLS] Drive-free progress protection: package the current
# checkpoint state into a compact ZIP and download it to your computer;
# after a destroyed runtime, re-upload it with the next cell and resume.
# Bundles contain ONLY checkpoints + compatibility metadata — never the
# cloned repository, .git, dataset copies, caches, or credentials.
import hashlib, json, time, zipfile
import numpy as np

CKPT_BUNDLE_SCHEMA = 1

def _ckpt_stage_summary():
    out = {}
    for stage, path in (('PASS1', PASS1_CKPT), ('PASS2', PASS2_CKPT)):
        if path.exists():
            with np.load(path, allow_pickle=False) as z:
                out[stage] = {
                    'models_done': int(z['done']) if 'done' in z.files
                                   else int(z['models_done']),
                    'next_k': int(z['next_k']),
                    'next_rank': int(z['next_rank']),
                    'digest': str(z['digest']),
                }
    return out

def export_checkpoint_bundle():
    """Create WORK_DIR/checkpoint_bundle.zip from the current checkpoints."""
    stages = _ckpt_stage_summary()
    if not stages:
        print('no checkpoint files exist yet — nothing to export')
        return None
    manifest = dict(
        schema_version=CKPT_BUNDLE_SCHEMA, created_utc=time.strftime(
            '%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        parquet_sha256=PARQUET_SHA256, metadata_sha256=METADATA_SHA256,
        code_commit=RESOLVED_COMMIT, p=P, n=n, g=G,
        model_prior=list(MODEL_PRIOR), dtype=DTYPE,
        top_k=TOP_K_MODELS, grid_points=GRID_POINTS,
        compute_coef_densities=bool(COMPUTE_COEF_DENSITIES),
        stages=stages,
        note='resume compatibility is additionally enforced by the SHA-256 '
             'digest stored inside each .ckpt.npz')
    mpath = TMP_DIR / 'checkpoint_bundle_manifest.json'
    mpath.write_text(json.dumps(manifest, indent=2))
    zpath = WORK_DIR / 'checkpoint_bundle.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(mpath, arcname='checkpoint_bundle_manifest.json')
        for path in (PASS1_CKPT, PASS2_CKPT):
            if path.exists():
                z.write(path, arcname=path.name)
        sidecar = CKPT_DIR / 'pass1_sidecar.json'
        if sidecar.exists():
            z.write(sidecar, arcname=sidecar.name)
    sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
    print(f'bundle: {zpath} ({zpath.stat().st_size:,} B)\nsha256: {sha}')
    for stage, info in stages.items():
        print(f'  {stage}: {info["models_done"]:,} models done, '
              f'next k={info["next_k"]}, rank={info["next_rank"]:,}')
    return zpath

def download_checkpoint_bundle():
    """export_checkpoint_bundle() + browser download via Colab."""
    zpath = export_checkpoint_bundle()
    if zpath is None:
        return
    try:
        from google.colab import files
        files.download(str(zpath))
    except Exception as exc:
        print('browser download unavailable here:', exc)

print('checkpoint tools ready: call download_checkpoint_bundle() at any '
      'time to save progress to your computer (no Drive needed)')


In [ ]:
# [CHECKPOINT RESTORE — optional] Upload a previously downloaded
# checkpoint_bundle.zip and resume without any Google Drive account.
# Set the flag to True and run this cell; it validates the bundle and
# rejects anything incompatible instead of risking double-counting.
import hashlib, json, pathlib, zipfile
import numpy as np

RESTORE_CHECKPOINT_UPLOAD = False   # set True, run, and pick the .zip file

def _safe_extract_zip(zpath, dest):
    """Extract, refusing absolute paths and path traversal."""
    dest = dest.resolve()
    with zipfile.ZipFile(zpath) as z:
        for zi in z.infolist():
            name = zi.filename.replace('\\', '/')
            if name.startswith('/') or '..' in name.split('/'):
                raise ValueError(f'unsafe path in bundle: {zi.filename!r}')
            target = (dest / name).resolve()
            if not str(target).startswith(str(dest)):
                raise ValueError(f'path escapes destination: {zi.filename!r}')
        z.extractall(dest)

def restore_checkpoint_bundle(zpath):
    stage_dir = TMP_DIR / 'restore_upload'     # outside the git clone
    stage_dir.mkdir(parents=True, exist_ok=True)
    _safe_extract_zip(zpath, stage_dir)
    mpath = stage_dir / 'checkpoint_bundle_manifest.json'
    if not mpath.exists():
        raise ValueError('bundle has no checkpoint_bundle_manifest.json')
    man = json.loads(mpath.read_text())
    problems = []
    def need(field, expected):
        if man.get(field) != expected:
            problems.append(f'{field}: bundle {man.get(field)!r} '
                            f'!= run {expected!r}')
    need('schema_version', CKPT_BUNDLE_SCHEMA)
    need('parquet_sha256', PARQUET_SHA256)
    need('code_commit', RESOLVED_COMMIT)
    need('p', P); need('n', n); need('g', G)
    need('model_prior', list(MODEL_PRIOR)); need('dtype', DTYPE)
    need('top_k', TOP_K_MODELS); need('grid_points', GRID_POINTS)
    need('compute_coef_densities', bool(COMPUTE_COEF_DENSITIES))
    if problems:
        raise ValueError('INCOMPATIBLE checkpoint bundle — refusing to '
                         'restore:\n  ' + '\n  '.join(problems))
    restored = []
    for name in (PASS1_CKPT.name, PASS2_CKPT.name, 'pass1_sidecar.json'):
        src = stage_dir / name
        if src.exists():
            (CKPT_DIR / name).write_bytes(src.read_bytes())
            restored.append(name)
    print('restored:', ', '.join(restored))
    for stage, info in man['stages'].items():
        print(f'  {stage}: resumes after {info["models_done"]:,} models '
              f'(next k={info["next_k"]}, rank={info["next_rank"]:,}) — '
              'ranges before that point are never re-counted')
    print('resume: leave AUTO_RESUME = True and run the PASS cells; each '
          'checkpoint is re-verified against the data/prior SHA-256 digest '
          'before any model is scored')

if RESTORE_CHECKPOINT_UPLOAD:
    from google.colab import files
    up = files.upload()
    for fname in up:
        restore_checkpoint_bundle(pathlib.Path(fname))
else:
    print('restore idle (set RESTORE_CHECKPOINT_UPLOAD = True to upload a '
          'bundle after a destroyed runtime)')


In [ ]:
# [SMOKE TEST — MANDATORY] Reduced deterministic CPU/GPU parity on the
# canonical rows (p = SMOKE_P, validated in the local phase). The expensive
# cell refuses to run unless SMOKE_TEST_PASSED is True.
import numpy as np, time
from scipy.special import logsumexp
from gpubma.cpu.enumeration import enumerate_models
from gpubma.gpu.enumerator import enumerate_models_gpu
from gpubma.priors.model_priors import log_model_prior_function

assert INPUT_VALIDATED
t0 = time.perf_counter()
p_s = SMOKE_P
cols = [f'x{j}' for j in range(1, p_s + 1)]
Xs = DF[cols].to_numpy(np.float64)
Xs_r = Xs - Q @ (Q.T @ Xs)
lp_s, _ = log_model_prior_function(MODEL_PRIOR, p_s)
kw = dict(df_resid=n - 1, g=G, log_model_prior=lp_s,
          tss_norm=CONV['tss_norm'], k_always=2, compute_coefficients=True)
cpu = enumerate_models(Xs_r, Y_R, top_k=10, **kw)
gpu = enumerate_models_gpu(Xs_r, Y_R, top_k=10, keep_scores=True, **kw)

ATOL_SCORE, ATOL_PROB = 1e-9, 1e-10   # validated local-phase tolerances
rng = np.random.default_rng(SMOKE_RNG_SEED)
special = {'null': 0, 'full': (1 << p_s) - 1}
for j in range(p_s):
    special[f'singleton_{j}'] = 1 << j
k_c = p_s // 2
special['central_first'] = (1 << k_c) - 1
special['central_last'] = ((1 << k_c) - 1) << (p_s - k_c)
for i, m in enumerate(rng.integers(0, 1 << p_s, SMOKE_N_RANDOM_MODELS)):
    special[f'random_{i}'] = int(m)

gpu_pmp = np.exp(gpu['log_scores'] - logsumexp(gpu['log_scores']))
SMOKE = {
    'model_count': (cpu['n_models_evaluated'] == 2**p_s
                    and gpu['n_models_evaluated'] == 2**p_s),
    'all_scores': float(np.abs(cpu['log_scores'] - gpu['log_scores']).max()),
    'special_scores': max(abs(float(cpu['log_scores'][m] - gpu['log_scores'][m]))
                          for m in special.values()),
    'log_normalizer': abs(cpu['log_normalizer'] - gpu['log_normalizer']),
    'pmp': float(np.abs(cpu['pmp'] - gpu_pmp).max()),
    'pip': float(np.abs(cpu['pip'] - gpu['pip']).max()),
    'size_dist': float(np.abs(cpu['size_distribution']
                              - gpu['size_distribution']).max()),
    'coef_mean': float(np.abs(cpu['coef_mean'] - gpu['coef_mean']).max()),
    'coef_sd': float(np.abs(cpu['coef_sd'] - gpu['coef_sd']).max()),
    'top10_masks_equal': ([m['mask'] for m in cpu['top_models']]
                          == [m['mask'] for m in gpu['top_models']]),
    'finite': bool(np.isfinite(gpu['log_scores']).all()),
}
SMOKE_TEST_PASSED = bool(
    SMOKE['model_count'] and SMOKE['top10_masks_equal'] and SMOKE['finite']
    and SMOKE['all_scores'] < ATOL_SCORE
    and SMOKE['special_scores'] < ATOL_SCORE
    and SMOKE['log_normalizer'] < ATOL_SCORE
    and SMOKE['pmp'] < ATOL_PROB and SMOKE['pip'] < ATOL_PROB
    and SMOKE['size_dist'] < ATOL_PROB
    and SMOKE['coef_mean'] < ATOL_SCORE and SMOKE['coef_sd'] < ATOL_SCORE)
T_SMOKE_S = time.perf_counter() - t0
print(f'smoke test p={p_s}: {2**p_s:,} models, {len(special)} named IDs, '
      f'{T_SMOKE_S:.1f} s')
for k, v in SMOKE.items():
    print(f'  {k:16s}: {v if isinstance(v, bool) else f"{v:.3e}"}')
print('SMOKE_TEST_PASSED =', SMOKE_TEST_PASSED)
if not SMOKE_TEST_PASSED:
    raise AssertionError('CPU/GPU parity smoke test FAILED — do not run the '
                         'full enumeration; investigate first.')


## EXPENSIVE A100 RUN — EXACT ENUMERATION OF 1,073,741,824 MODELS

The next cell is the intentional, expensive step. Before running it:

- confirm the runtime is an **A100** (cell 2 output);
- confirm the smoke test **PASSED** (previous cell);
- mount Google Drive first (`USE_GOOGLE_DRIVE = True`) if you want
  checkpoint persistence across disconnects — strongly recommended;
- it consumes Colab compute units; the run is exact (all 2^30 models) and
  cannot be made cheaper without changing the science;
- then set **`RUN_FULL_EXACT_P30 = True`** in the config cell and re-run
  the config cell, then run the cell below.

The cell never auto-starts from bootstrap, installation, or smoke cells.


In [ ]:
# [PASS1 — EXPENSIVE A100 RUN] Exact enumeration of model IDs 0 .. 2^30 - 1
# (1,073,741,824 models) with the validated GPU enumerator: streamed direct
# batching, on-device combinadic unranking, batched Cholesky on the
# residualized Gram (FWL/shrink formulation), deterministic streaming
# log-sum-exp reductions, SHA-256-gated atomic checkpoints. No sampling,
# no pruning, no truncation, no early stopping.
import json, time
import numpy as np
import torch

if not RUN_FULL_EXACT_P30:
    print('RUN_FULL_EXACT_P30 is False — the expensive run is DISABLED.\n'
          'To start it intentionally: set RUN_FULL_EXACT_P30 = True in the '
          'config cell, re-run that cell, then re-run this one.')
    PASS1_RESULT = None
elif PASS1_JSON.exists():
    print('PASS1 results already on disk — skipping enumeration (idempotent)')
    PASS1_RESULT = json.loads(PASS1_JSON.read_text())
else:
    assert INPUT_VALIDATED, 'input validation must pass first'
    if not SMOKE_TEST_PASSED:
        if EXPERT_OVERRIDE_SKIP_SMOKE_GATE:
            print('*** WARNING: smoke-test gate BYPASSED by expert override. '
                  'You are running a billion-model job without CPU/GPU '
                  'parity evidence. ***')
        else:
            raise RuntimeError('smoke test has not passed; refusing to start '
                               '(EXPERT_OVERRIDE_SKIP_SMOKE_GATE=False)')
    from gpubma.gpu.enumerator import enumerate_models_gpu

    free_b, _ = torch.cuda.mem_get_info()
    vram_budget = int(VRAM_BUDGET_FRACTION * free_b)
    total_chunks_est = (N_MODELS + MAX_CHUNK - 1) // MAX_CHUNK
    (CKPT_DIR / 'pass1_sidecar.json').write_text(json.dumps({
        'stage': 'PASS1', 'commit': RESOLVED_COMMIT,
        'parquet_sha256': PARQUET_SHA256, 'metadata_sha256': METADATA_SHA256,
        'p': P, 'n': n, 'g': G, 'model_prior': list(MODEL_PRIOR),
        'dtype': DTYPE, 'max_chunk': MAX_CHUNK, 'top_k': TOP_K_MODELS,
        'note': 'compatibility itself is enforced by the SHA-256 digest '
                'inside the .ckpt.npz (sufficient statistics + prior config)',
    }, indent=2))

    t_wall = time.time()
    state = {'last_done': 0, 'last_t': t_wall}
    def show(info):
        now = time.time()
        recent = ((info['models_done'] - state['last_done'])
                  / max(now - state['last_t'], 1e-9))
        state['last_done'], state['last_t'] = info['models_done'], now
        overall = info['models_per_second']
        eta = (info['models_total'] - info['models_done']) / max(overall, 1e-9)
        chunk_i = info['models_done'] // MAX_CHUNK
        print(f"[PASS1 {info['fraction']:7.2%}] layer k={info['current_size']:2d} "
              f"chunk~{chunk_i:,}/{total_chunks_est:,}  "
              f"{info['models_done']:,}/{info['models_total']:,}  "
              f"elapsed {info['elapsed_s']:,.0f}s  "
              f"recent {recent:,.0f}/s  overall {overall:,.0f}/s  "
              f"ETA {eta:,.0f}s  GPU {torch.cuda.memory_allocated()/2**30:.2f}"
              f"/{torch.cuda.max_memory_allocated()/2**30:.2f} GiB peak",
              flush=True)

    X30_r = X_r  # residualized candidates from the validation cell
    resume = AUTO_RESUME and PASS1_CKPT.exists()
    print('resuming from checkpoint' if resume else 'starting fresh')
    res = enumerate_models_gpu(
        X30_r, Y_R, g=G, log_model_prior=LOG_PRIOR, top_k=TOP_K_MODELS,
        vram_budget_bytes=vram_budget, max_chunk=MAX_CHUNK,
        checkpoint_path=PASS1_CKPT, checkpoint_every_s=CHECKPOINT_EVERY_S,
        resume=resume, progress_every_s=PROGRESS_EVERY_S, progress=show,
        **CONV)
    T_PASS1_WALL_S = time.time() - t_wall
    assert res['n_models_evaluated'] == N_MODELS == 1_073_741_824
    PASS1_RESULT = {
        'n_models_evaluated': res['n_models_evaluated'],
        'log_normalizer': res['log_normalizer'],
        'pip': np.asarray(res['pip']).tolist(),
        'mean_model_size': res['mean_model_size'],
        'size_distribution': np.asarray(res['size_distribution']).tolist(),
        'coef_mean': np.asarray(res['coef_mean']).tolist(),
        'coef_sd': np.asarray(res['coef_sd']).tolist(),
        'top_models': res['top_models'],
        'tss_residualized': res['tss_residualized'],
        'normalization_check': res['normalization_check'],
        'runtime': res['runtime'],
        'wall_s_this_session': T_PASS1_WALL_S,
        'data_load_s': T_DATA_LOAD_S,
    }
    PASS1_JSON.write_text(json.dumps(PASS1_RESULT, indent=2))
    print(f"PASS1 complete: {res['n_models_evaluated']:,} models in "
          f"{res['runtime']['elapsed_s']:,.1f}s cumulative "
          f"({res['runtime']['models_per_second']:,.0f} models/s), "
          f"peak GPU {res['runtime']['peak_gpu_memory_bytes']/2**30:.2f} GiB")


In [ ]:
# [PASS1 VALIDATION] Exact-count and posterior sanity assertions.
import numpy as np

if PASS1_RESULT is None:
    print('PASS1 has not produced results in this session — nothing to validate')
else:
    r = PASS1_RESULT
    assert r['n_models_evaluated'] == 1_073_741_824 == (1 << 30)
    sd = np.asarray(r['size_distribution'])
    pip = np.asarray(r['pip'])
    assert abs(sd.sum() - 1.0) < 1e-9, f'size distribution sums to {sd.sum()!r}'
    assert r['normalization_check']['pip_max_overshoot'] <= 1e-12
    assert ((pip >= 0.0) & (pip <= 1.0)).all()
    assert np.isfinite(r['log_normalizer'])
    assert r['runtime']['precision'] == 'float64'
    ms = float(np.arange(P + 1) @ sd)
    assert abs(ms - r['mean_model_size']) < 1e-9
    print(f"VALIDATED: exactly {r['n_models_evaluated']:,} models; "
          f"size-dist sum = {sd.sum():.15f}; PIPs in [0, 1]; "
          f"mean model size = {r['mean_model_size']:.6f}")


In [ ]:
# [PASS2] Bounded-memory exact second sweep over ALL 2^30 models (weights =
# exact global PMPs from the saved PASS1 normalizer). Produces what PASS1's
# O(p) accumulators cannot: the top-K model table (K = TOP_K_MODELS), the
# per-family joint posterior (true/proxy both/only/neither), the EXACT
# posterior rank of the true model, score extrema / non-finite counts,
# coefficient sign probabilities, and (optionally) exact conditional
# coefficient density grids. Deterministic, checkpointed, float64.
#
# Sign probabilities use the Gaussian limit of the t_(n-1) conditional
# posterior (n - 1 = 1999): absolute error < 1e-3, labelled 'approx' in the
# output schema. Everything else in this pass is exact.
import hashlib, json, math, time
import numpy as np
import torch

if PASS1_RESULT is None or not RUN_PASS2:
    print('PASS2 skipped (needs PASS1 results and RUN_PASS2=True)')
    PASS2_DONE = False
else:
    from gpubma.gpu.enumerator import (binomial_table, unrank_combinations,
                                       _save_checkpoint, _load_checkpoint)
    dev = torch.device('cuda')
    LOG_NORM = float(PASS1_RESULT['log_normalizer'])
    PIP1 = np.asarray(PASS1_RESULT['pip'])
    CM1 = np.asarray(PASS1_RESULT['coef_mean'])
    CS1 = np.asarray(PASS1_RESULT['coef_sd'])
    Zxx_np, Zxy_np = X_r.T @ X_r, X_r.T @ Y_R
    tss = float(Y_R @ Y_R)
    tss_norm, k_alw, dfree = CONV['tss_norm'], CONV['k_always'], float(n - 1)
    ess_always = tss_norm - tss
    lp_np = np.array([LOG_PRIOR(k) for k in range(P + 1)])
    s_g = G / (1.0 + G)
    log1pg = math.log1p(G)
    tiny = float(np.finfo(np.float64).tiny)
    c_t = (math.lgamma((dfree + 1) / 2) - math.lgamma(dfree / 2)
           - 0.5 * math.log(dfree * math.pi))
    PASS2_CHUNK = 8192

    # exact log score of the TRUE model (x1..x15) computed up front
    from scipy.linalg import cho_factor, cho_solve
    idx_t = np.arange(15)
    cft = cho_factor(Zxx_np[np.ix_(idx_t, idx_t)], lower=True)
    ess_t = float(Zxy_np[idx_t] @ cho_solve(cft, Zxy_np[idx_t]))
    omr_t = max((tss - ess_t) / tss_norm, tiny)
    TRUE_SCORE = (0.5 * (dfree - 15 - k_alw) * log1pg
                  - 0.5 * dfree * math.log1p(G * omr_t) + lp_np[15])

    def grid_np():
        pip_safe = np.maximum(PIP1, 1e-300)
        cm = CM1 / pip_safe
        cv = np.maximum((CS1**2 + CM1**2) / pip_safe - cm**2, 1e-30)
        cs = np.sqrt(cv)
        return np.linspace(cm - 8 * cs, cm + 8 * cs, GRID_POINTS, axis=1)

    GRID = grid_np()
    h = hashlib.sha256()
    for arr in (Zxx_np, Zxy_np, lp_np, GRID):
        h.update(np.ascontiguousarray(arr, np.float64).tobytes())
    h.update(np.array([tss, tss_norm, dfree, G, k_alw, LOG_NORM, TRUE_SCORE,
                       TOP_K_MODELS, GRID_POINTS,
                       float(COMPUTE_COEF_DENSITIES)], np.float64).tobytes())
    digest = h.hexdigest()

    Zxx = torch.from_numpy(Zxx_np).to(dev)
    Zxy = torch.from_numpy(Zxy_np).to(dev)
    grid_t = torch.from_numpy(GRID).to(dev)
    binom_np = binomial_table(P)
    binom = torch.from_numpy(binom_np).to(dev)
    jbits = torch.arange(P, dtype=torch.int64, device=dev)

    st0 = dict(
        dens=torch.zeros(P, GRID_POINTS, dtype=torch.float64, device=dev),
        top_s=torch.full((TOP_K_MODELS,), -math.inf, dtype=torch.float64,
                         device=dev),
        top_m=torch.zeros(TOP_K_MODELS, dtype=torch.int64, device=dev),
        sum_w=torch.zeros((), dtype=torch.float64, device=dev),
        fam=torch.zeros(15, 3, dtype=torch.float64, device=dev),  # both/T-only/P-only
        sign_num=torch.zeros(P, dtype=torch.float64, device=dev),
        n_higher=torch.zeros((), dtype=torch.float64, device=dev),
        n_tied=torch.zeros((), dtype=torch.float64, device=dev),
        mass_ge=torch.zeros((), dtype=torch.float64, device=dev),
        smin=torch.tensor(math.inf, dtype=torch.float64, device=dev),
        smax=torch.tensor(-math.inf, dtype=torch.float64, device=dev),
        nonfinite=torch.zeros((), dtype=torch.float64, device=dev),
    )
    done, k0, r0 = 0, 0, 0
    if AUTO_RESUME and PASS2_CKPT.exists():
        cp = _load_checkpoint(PASS2_CKPT)
        assert str(cp['digest']) == digest, ('PASS2 checkpoint belongs to a '
                                             'different configuration; refusing')
        for key in st0:
            st0[key] = torch.from_numpy(np.atleast_1d(cp[key])).to(dev).reshape(
                st0[key].shape).clone()
        done, k0, r0 = int(cp['done']), int(cp['next_k']), int(cp['next_rank'])
        print(f'PASS2 resuming at k={k0}, rank={r0:,} ({done:,} models done)')

    def save_ckpt(nk, nr):
        state = {key: v.cpu().numpy() for key, v in st0.items()}
        state.update(digest=np.str_(digest), done=np.int64(done),
                     next_k=np.int64(nk), next_rank=np.int64(nr))
        _save_checkpoint(PASS2_CKPT, state)

    t0 = time.time()
    last_ck = last_pr = t0
    if PASS2_CKPT.exists() and done >= N_MODELS:
        print('PASS2 already complete in checkpoint')
    for k in range(k0, P + 1):
        total_k = int(binom_np[P, k])
        r = r0 if k == k0 else 0
        while r < total_k and done < N_MODELS:
            B = min(PASS2_CHUNK, total_k - r)
            if k == 0:
                omr = max(tss / tss_norm, tiny)
                sc = torch.tensor([0.5 * (dfree - k_alw) * log1pg
                                   - 0.5 * dfree * math.log1p(G * omr)
                                   + lp_np[0]], dtype=torch.float64, device=dev)
                masks = torch.zeros(1, dtype=torch.int64, device=dev)
                w = torch.exp(sc - LOG_NORM)
            else:
                ranks = torch.arange(r, r + B, dtype=torch.int64, device=dev)
                idx = unrank_combinations(ranks, k, binom, torch)
                Z = Zxx[idx.unsqueeze(2), idx.unsqueeze(1)]
                b = Zxy[idx].unsqueeze(-1)
                L = torch.linalg.cholesky(Z)
                u = torch.linalg.solve_triangular(L, b, upper=False)
                ess = (u.squeeze(-1) ** 2).sum(dim=1)
                omr = torch.clamp((tss - ess) / tss_norm, min=tiny)
                sc = (0.5 * (dfree - k - k_alw) * log1pg
                      - 0.5 * dfree * torch.log1p(G * omr) + lp_np[k])
                masks = (torch.ones_like(idx) << idx).sum(dim=1)
                w = torch.exp(sc - LOG_NORM)
                beta_hat = torch.linalg.solve_triangular(
                    L.transpose(1, 2), u, upper=True).squeeze(-1)
                zinv = torch.cholesky_inverse(L).diagonal(dim1=-2, dim2=-1)
                b_gam = tss_norm - s_g * (ess_always + ess)
                loc = s_g * beta_hat
                scale = torch.sqrt((b_gam / dfree).unsqueeze(1) * s_g * zinv)
                onehot = torch.nn.functional.one_hot(idx, P).to(torch.float64)
                # sign prob (normal limit of t_1999): Phi(loc/scale)
                phi = 0.5 * (1.0 + torch.erf(loc / scale / math.sqrt(2.0)))
                st0['sign_num'] += torch.einsum('bkp,bk->p', onehot,
                                                w.unsqueeze(1) * phi)
                if COMPUTE_COEF_DENSITIES:
                    xg = grid_t[idx]
                    zz = (xg - loc.unsqueeze(-1)) / scale.unsqueeze(-1)
                    logpdf = (c_t - torch.log(scale).unsqueeze(-1)
                              - 0.5 * (dfree + 1) * torch.log1p(zz * zz / dfree))
                    st0['dens'] += torch.einsum(
                        'bkp,bkg->pg', onehot,
                        w.unsqueeze(-1).unsqueeze(-1) * torch.exp(logpdf))
            bits = ((masks.unsqueeze(1) >> jbits) & 1).to(torch.float64)
            tb, pb = bits[:, :15], bits[:, 15:]
            st0['fam'][:, 0] += (w.unsqueeze(1) * tb * pb).sum(dim=0)
            st0['fam'][:, 1] += (w.unsqueeze(1) * tb * (1 - pb)).sum(dim=0)
            st0['fam'][:, 2] += (w.unsqueeze(1) * (1 - tb) * pb).sum(dim=0)
            st0['sum_w'] += w.sum()
            st0['n_higher'] += (sc > TRUE_SCORE).sum()
            st0['n_tied'] += (sc == TRUE_SCORE).sum()
            st0['mass_ge'] += (w * (sc >= TRUE_SCORE)).sum()
            finite = torch.isfinite(sc)
            st0['nonfinite'] += (~finite).sum()
            st0['smin'] = torch.minimum(st0['smin'], sc[finite].min())
            st0['smax'] = torch.maximum(st0['smax'], sc[finite].max())
            cs_ = torch.cat([st0['top_s'], sc])
            cm_ = torch.cat([st0['top_m'], masks])
            best = torch.topk(cs_, k=TOP_K_MODELS)
            st0['top_s'], st0['top_m'] = best.values, cm_[best.indices]
            done += B
            r += B
            now = time.time()
            if now - last_pr >= PROGRESS_EVERY_S:
                rate = done / max(now - t0, 1e-9)
                print(f'[PASS2 {done / N_MODELS:7.2%}] layer k={k:2d}  '
                      f'{done:,}/{N_MODELS:,}  {rate:,.0f}/s  '
                      f'ETA {(N_MODELS - done) / max(rate, 1e-9):,.0f}s',
                      flush=True)
                last_pr = now
            if now - last_ck >= CHECKPOINT_EVERY_S:
                save_ckpt(k if r < total_k else k + 1, r if r < total_k else 0)
                last_ck = now
    assert done == N_MODELS, f'PASS2 incomplete: {done:,}'
    total_w = float(st0['sum_w'].cpu())
    assert abs(total_w - 1.0) < 1e-9, f'sum of exact PMPs = {total_w!r}'
    save_ckpt(P + 1, 0)
    T_PASS2_S = time.time() - t0
    order = torch.argsort(st0['top_s'], descending=True)
    TOP_SCORES = st0['top_s'][order].cpu().numpy()
    TOP_MASKS = st0['top_m'][order].cpu().numpy()
    FAM = st0['fam'].cpu().numpy()          # (15, 3): both, true-only, proxy-only
    SIGN_NUM = st0['sign_num'].cpu().numpy()
    DENS = st0['dens'].cpu().numpy() if COMPUTE_COEF_DENSITIES else None
    TRUE_RANK = int(float(st0['n_higher'].cpu())) + 1
    TRUE_TIES = int(float(st0['n_tied'].cpu())) - 1
    MASS_GE_TRUE = float(st0['mass_ge'].cpu())
    SCORE_MIN = float(st0['smin'].cpu())
    SCORE_MAX = float(st0['smax'].cpu())
    N_NONFINITE = int(float(st0['nonfinite'].cpu()))
    PASS2_DONE = True
    print(f'PASS2 complete: {done:,} models, sum PMP = {total_w:.15f}, '
          f'{T_PASS2_S:,.0f} s this session; true-model exact rank = '
          f'{TRUE_RANK:,} (ties: {TRUE_TIES})')


In [ ]:
# [ARTIFACTS] Compact canonical outputs + SHA-256 for every file.
import hashlib, json, time
import numpy as np
import pandas as pd

if PASS1_RESULT is None or not PASS2_DONE:
    print('artifacts skipped — PASS1 and PASS2 must both be complete')
else:
    import torch
    def sha(p):
        return hashlib.sha256(p.read_bytes()).hexdigest()
    def popcount(m):
        return bin(int(m)).count('1')

    lognorm = PASS1_RESULT['log_normalizer']
    pip = np.asarray(PASS1_RESULT['pip'])
    cm, cs = (np.asarray(PASS1_RESULT['coef_mean']),
              np.asarray(PASS1_RESULT['coef_sd']))
    sd = np.asarray(PASS1_RESULT['size_distribution'])
    names = [f'x{j}' for j in range(1, 31)]
    pmap = {m['proxy']: m for m in META['proxy_mappings']}

    # B. variable-level posterior table
    pip_safe = np.maximum(pip, 1e-300)
    cond_mean = cm / pip_safe
    cond_var = np.maximum((cs**2 + cm**2) / pip_safe - cond_mean**2, 0.0)
    var_rows = []
    for i, v in enumerate(names):
        is_proxy = i >= 15
        src = (f"{pmap[v]['primary_true_variable']}+"
               f"{pmap[v]['secondary_partner']}") if is_proxy else 'latent block'
        var_rows.append(dict(
            variable=v, var_class='proxy' if is_proxy else 'true',
            source_mapping=src, pip=float(pip[i]),
            coef_mean=float(cm[i]), coef_sd=float(cs[i]),
            cond_incl_mean=float(cond_mean[i]), cond_incl_var=float(cond_var[i]),
            sign_prob_positive_given_incl_approx=float(SIGN_NUM[i] / pip_safe[i]),
        ))
    pip_df = pd.DataFrame(var_rows)
    f_pip = RESULTS_DIR / 'panel_30_center15_pip.parquet'
    pip_df.to_parquet(f_pip, index=False)

    # C. family-level posterior (canonical mapping: proxy x{15+j} has primary
    # x{j} AND a same-block secondary partner — preserved, not one-to-one)
    fam_rows = []
    for j in range(15):
        both, tonly, ponly = (float(FAM[j, 0]), float(FAM[j, 1]),
                              float(FAM[j, 2]))
        neither = 1.0 - both - tonly - ponly
        m = pmap[f'x{16 + j}']
        fam_rows.append(dict(
            family=j + 1, true_variable=f'x{j + 1}', proxy_variable=f'x{16 + j}',
            proxy_secondary_partner=m['secondary_partner'],
            p_true_included=tonly + both, p_proxy_included=ponly + both,
            p_both=both, p_neither=neither, p_at_least_one=1.0 - neither,
            p_true_only=tonly, p_proxy_only=ponly))
    fam_df = pd.DataFrame(fam_rows)
    f_fam = RESULTS_DIR / 'panel_30_center15_family_posterior.parquet'
    fam_df.to_parquet(f_fam, index=False)

    # D. model-size posterior
    cum = np.cumsum(sd)
    k_mode = int(np.argmax(sd))
    k_median = int(np.searchsorted(cum, 0.5))
    lo = int(np.searchsorted(cum, 0.025))
    hi = int(np.searchsorted(cum, 0.975))
    size_df = pd.DataFrame({'k': np.arange(31), 'posterior': sd,
                            'cumulative': cum})
    f_size = RESULTS_DIR / 'panel_30_center15_model_size.parquet'
    size_df.to_parquet(f_size, index=False)

    # E. top models
    finite = np.isfinite(TOP_SCORES)
    tm_scores, tm_masks = TOP_SCORES[finite], TOP_MASKS[finite]
    tm_pmp = np.exp(tm_scores - lognorm)
    rows = []
    for rank, (s_, m_) in enumerate(zip(tm_scores, tm_masks), start=1):
        m_ = int(m_)
        inc = [j for j in range(30) if (m_ >> j) & 1]
        rows.append(dict(
            rank=rank, model_id=m_, model_size=len(inc),
            included=','.join(names[j] for j in inc),
            log_score=float(s_), log_posterior_weight=float(s_ - lognorm),
            pmp=float(np.exp(s_ - lognorm)),
            n_true=sum(1 for j in inc if j < 15),
            n_proxy=sum(1 for j in inc if j >= 15),
            hamming_from_true_model=popcount(m_ ^ TRUE_MASK)))
    top_df = pd.DataFrame(rows)
    top_df['cumulative_pmp'] = top_df['pmp'].cumsum()
    f_top = RESULTS_DIR / 'panel_30_center15_top_models.parquet'
    top_df.to_parquet(f_top, index=False)
    conc = {str(kk): float(top_df['pmp'].iloc[:kk].sum())
            for kk in (1, 10, 100, 1000, 10000) if kk <= len(top_df)}

    # F. exact true-model diagnostics
    top1_mask = int(tm_masks[0])
    true_diag = dict(
        model_id=int(TRUE_MASK), model_size=15, log_score=float(TRUE_SCORE),
        pmp=float(np.exp(TRUE_SCORE - lognorm)),
        exact_posterior_rank=TRUE_RANK, score_ties=TRUE_TIES,
        cumulative_pmp_through_rank=MASS_GE_TRUE,
        log_score_gap_from_top=float(tm_scores[0] - TRUE_SCORE),
        hamming_from_top_model=popcount(TRUE_MASK ^ top1_mask),
        note='exact rank counted over all 2^30 models in PASS2; the true '
             'model is NOT required to rank first in this proxy design')
    f_true = RESULTS_DIR / 'panel_30_center15_true_model.json'
    f_true.write_text(json.dumps(true_diag, indent=2))

    # G. posterior diagnostics
    validation = dict(
        total_models_accounted=int(N_MODELS),
        pmp_normalization_error=abs(float(st0['sum_w'].cpu()) - 1.0),
        size_posterior_normalization_error=abs(float(sd.sum()) - 1.0),
        pip_max_overshoot=PASS1_RESULT['normalization_check']['pip_max_overshoot'],
        min_finite_log_score=SCORE_MIN, max_finite_log_score=SCORE_MAX,
        n_nonfinite_scores=N_NONFINITE, n_rejected_models=0,
        cholesky_failures=0,
        cholesky_note='any batched Cholesky failure raises and aborts the '
                      'run, so completion implies zero failures',
        top_two_gap=float(tm_scores[0] - tm_scores[1]),
        near_ties_within_1e9_of_top=int((tm_scores[0] - tm_scores < 1e-9).sum()),
        numerical_warnings=[])
    f_val = RESULTS_DIR / 'panel_30_center15_validation.json'
    f_val.write_text(json.dumps(validation, indent=2))

    if COMPUTE_COEF_DENSITIES and DENS is not None:
        f_dens = RESULTS_DIR / 'panel_30_center15_coef_densities.npz'
        np.savez_compressed(f_dens, grid=GRID, weighted_density=DENS,
                            conditional_density=DENS / pip_safe[:, None],
                            pip=pip, df=float(n - 1))

    # exact summary + A. run manifest (manifest hashed last, includes outputs)
    summary = dict(
        dataset='panel_30_center15', n_models=int(N_MODELS),
        log_normalizer=lognorm, mean_model_size=PASS1_RESULT['mean_model_size'],
        size_mode=k_mode, size_median=k_median,
        size_credible_interval_95=[lo, hi],
        size_ci_method='central quantile interval from the cumulative size '
                       'posterior (first k with cum >= 0.025 / 0.975)',
        pip={v['variable']: v['pip'] for v in var_rows},
        top_mass_concentration=conc, true_model=true_diag,
        family_posterior=fam_rows)
    f_sum = RESULTS_DIR / 'panel_30_center15_exact_summary.json'
    f_sum.write_text(json.dumps(summary, indent=2))

    md = ['# panel_30_center15 — exact 2^30 BMA results (A100)', '',
          f'- models: {N_MODELS:,} (exact, verified)',
          f'- PASS1: {PASS1_RESULT["runtime"]["elapsed_s"]:,.1f} s cumulative '
          f'({PASS1_RESULT["runtime"]["models_per_second"]:,.0f} models/s); '
          f'PASS2: {T_PASS2_S:,.0f} s (this session)',
          f'- peak GPU: '
          f'{PASS1_RESULT["runtime"]["peak_gpu_memory_bytes"]/2**30:.2f} GiB',
          f'- posterior mean model size {PASS1_RESULT["mean_model_size"]:.3f}, '
          f'mode {k_mode}, median {k_median}, 95% CI [{lo}, {hi}]',
          f'- true model: rank {TRUE_RANK:,}, PMP {true_diag["pmp"]:.3e}, '
          f'gap from top {true_diag["log_score_gap_from_top"]:.4f}',
          f'- top-mass concentration: ' + ', '.join(
              f'top {kk}: {v:.4f}' for kk, v in conc.items()), '',
          '| variable | class | PIP |', '|---|---|---|']
    md += [f"| {v['variable']} | {v['var_class']} | {v['pip']:.4f} |"
           for v in var_rows]
    f_md = REPORTS_DIR / 'panel_30_center15_a100_exact.md'
    f_md.write_text('\n'.join(md) + '\n', encoding='utf-8')

    outputs = [f_sum, f_pip, f_fam, f_size, f_top, f_true, f_val, f_md]
    if COMPUTE_COEF_DENSITIES and DENS is not None:
        outputs.append(f_dens)
    out_hashes = {p.name: sha(p) for p in outputs}
    manifest = dict(
        dataset_rel=DATASET_REL, parquet_sha256=PARQUET_SHA256,
        metadata_sha256=METADATA_SHA256, code_commit=RESOLVED_COMMIT,
        repository=GITHUB_REPO_URL,
        environment=dict(python=sys.version.split()[0],
                         torch=torch.__version__, cuda=torch.version.cuda,
                         driver=DRIVER_VERSION),
        hardware=dict(gpu=GPU_NAME, vram_gib=GPU_VRAM_GIB),
        config=dict(p=P, n=n, g=G, model_prior=list(MODEL_PRIOR), dtype=DTYPE,
                    max_chunk=MAX_CHUNK, top_k=TOP_K_MODELS,
                    grid_points=GRID_POINTS,
                    checkpoint_every_s=CHECKPOINT_EVERY_S,
                    densities=bool(COMPUTE_COEF_DENSITIES)),
        timings=dict(data_load_s=PASS1_RESULT.get('data_load_s'),
                     smoke_s=globals().get('T_SMOKE_S'),
                     pass1_cumulative_s=PASS1_RESULT['runtime']['elapsed_s'],
                     pass1_models_per_s=PASS1_RESULT['runtime']['models_per_second'],
                     pass2_this_session_s=T_PASS2_S),
        total_models=int(N_MODELS), output_sha256=out_hashes)
    f_man = RESULTS_DIR / 'panel_30_center15_run_manifest.json'
    f_man.write_text(json.dumps(manifest, indent=2))
    ARTIFACTS = outputs + [f_man]
    print('artifacts written:')
    for p in ARTIFACTS:
        print(f'  {p.name:48s} {p.stat().st_size:>10,} B  '
              f'{sha(p)[:16]}...')


In [ ]:
# [DISPLAY] Concise final summaries (no huge tables).
import numpy as np
import pandas as pd

if PASS1_RESULT is None or not PASS2_DONE:
    print('displays skipped — complete PASS1 and PASS2 first')
else:
    pd.set_option('display.width', 120)
    print('=== PIPs (true x1-x15 | proxy x16-x30) ===')
    print(pip_df[['variable', 'var_class', 'pip', 'coef_mean',
                  'coef_sd']].to_string(index=False,
                                        float_format=lambda v: f'{v: .4f}'))
    print('\n=== family posterior recovery (per true/proxy pair) ===')
    print(fam_df[['true_variable', 'proxy_variable', 'p_true_included',
                  'p_proxy_included', 'p_both', 'p_neither',
                  'p_true_only', 'p_proxy_only']].to_string(
        index=False, float_format=lambda v: f'{v: .4f}'))
    print('\n=== model-size posterior (nonzero region) ===')
    nz = size_df[size_df['posterior'] > 1e-6]
    print(nz.to_string(index=False, float_format=lambda v: f'{v: .5f}'))
    print('\n=== top 10 models ===')
    print(top_df.head(10)[['rank', 'model_size', 'n_true', 'n_proxy',
                           'hamming_from_true_model', 'pmp',
                           'cumulative_pmp']].to_string(index=False))
    print('\n=== concentration ===', conc)
    print('\n=== exact true-model diagnostics ===')
    for kk, vv in true_diag.items():
        print(f'  {kk}: {vv}')
    r1 = PASS1_RESULT['runtime']
    print(f"\nPASS1 {r1['elapsed_s']:,.1f}s cumulative "
          f"({r1['models_per_second']:,.0f} models/s, "
          f"{r1['chunks']:,} chunks), PASS2 {T_PASS2_S:,.0f}s, "
          f"peak GPU {r1['peak_gpu_memory_bytes']/2**30:.2f} GiB, "
          f"total models {PASS1_RESULT['n_models_evaluated']:,}")


In [ ]:
# [EXPORT] Compact results ZIP, always built under /content and downloaded
# through Colab regardless of storage mode. Final artifacts ONLY — no
# checkpoints, no caches, no temporary chunks, no cloned repository, no
# .git, no dataset copy, no credentials, no prompts.
import hashlib, pathlib, shutil, zipfile

if PASS1_RESULT is None or not PASS2_DONE:
    print('export skipped — nothing complete to export')
else:
    zpath = pathlib.Path('/content') / 'panel_30_center15_exact_results.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in ARTIFACTS:
            z.write(p, arcname=f'{p.parent.name}/{p.name}')
    print('ZIP contents:')
    with zipfile.ZipFile(zpath) as z:
        for zi in z.infolist():
            print(f'  {zi.filename:56s} {zi.file_size:>10,} B')
    zip_sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
    print(f'\n{zpath.name}: {zpath.stat().st_size:,} B\nsha256: {zip_sha}')
    if USE_GOOGLE_DRIVE:
        drive_copy = OUT_DIR / zpath.name
        shutil.copy2(zpath, drive_copy)
        print('copy saved to Drive:', drive_copy)
    try:
        from google.colab import files
        files.download(str(zpath))
    except Exception as exc:
        print('browser download unavailable here:', exc)


## Resuming after an interruption

**Same runtime still alive** (kernel restart, cell rerun): run all cells
again with `RUN_FULL_EXACT_P30 = True` — local checkpoints under
`/content` are found automatically and the run continues (idempotent;
finished stages are skipped).

**Runtime destroyed — default free mode** (`USE_GOOGLE_DRIVE = False`):
local checkpoints are gone with the runtime. If you downloaded a
checkpoint bundle beforehand (`download_checkpoint_bundle()`), reopen the
notebook, run the cells through the restore cell, set
`RESTORE_CHECKPOINT_UPLOAD = True`, upload the bundle, then continue with
`RUN_FULL_EXACT_P30 = True`. The bundle is validated (dataset SHA-256,
pinned commit, p, n, priors, dtype, schema) and rejected if incompatible,
so already-processed model ranges are never double-counted.

**Runtime destroyed — optional Drive mode** (`USE_GOOGLE_DRIVE = True`):
reopen and Run all; PASS1/PASS2 resume automatically from the newest
matching Drive checkpoint (at most ~`CHECKPOINT_EVERY_S` of work lost).

In every mode, checkpoints are SHA-256-gated — a checkpoint from different
data, priors, or configuration is rejected instead of silently reused —
and `*.ckpt.npz` files live outside the git clone, are git-ignored, and
are excluded from the results ZIP.
